# 草稿1

In [4]:
chr(0)

'\x00'

In [5]:
print(chr(0))

 


In [6]:
chr(0).__repr__()

"'\\x00'"

In [7]:
'this is a test' + chr(0) + 'string'

'this is a test\x00string'

In [8]:
print('this is a test' + chr(0) + 'string')

this is a test string


In [4]:
test_string = "hello! こんにちは!"
utf8_encoded = test_string.encode("utf-8")
print(utf8_encoded)
print(utf8_encoded.decode("utf-8"))
print(type(utf8_encoded))

b'hello! \xe3\x81\x93\xe3\x82\x93\xe3\x81\xab\xe3\x81\xa1\xe3\x81\xaf!'
hello! こんにちは!
<class 'bytes'>


In [7]:
for i in range(len(test_string)):
    print(test_string[i], utf8_encoded[i])
    
print(list(utf8_encoded))

h 104
e 101
l 108
l 108
o 111
! 33
  32
こ 227
ん 129
に 147
ち 227
は 130
! 147
[104, 101, 108, 108, 111, 33, 32, 227, 129, 147, 227, 130, 147, 227, 129, 171, 227, 129, 161, 227, 129, 175, 33]


In [24]:
print(len(test_string))
print(len(utf8_encoded))

13
23


In [103]:
max([("A", "B"), ("A", "C"), ("B", "ZZ"), ("BA", "A")])

('BA', 'A')

In [ ]:
import regex as re
PAT = r"""'(?:[sdmt]|ll|ve|re)| ?\p{L}+| ?\p{N}+| ?[^\s\p{L}\p{N}]+|\s+(?!\S)|\s+"""
re.findall(PAT, "some text that i'll pre-tokenize")

['some', ' text', ' that', ' i', "'ll", ' pre', '-', 'tokenize']

In [ ]:
# 下载数据并保存
from datasets import load_dataset
ds = load_dataset("roneneldan/TinyStories")
ds.save_to_disk('../data/TinyStories')

In [ ]:
# 加载数据
from datasets import load_from_disk
dataset = load_from_disk('../data/TinyStories')

In [16]:
dataset

DatasetDict({
    train: Dataset({
        features: ['text'],
        num_rows: 2119719
    })
    validation: Dataset({
        features: ['text'],
        num_rows: 21990
    })
})

# 草稿2：课程资料上的代码

In [ ]:
import os 
import sys
from abc import ABC
import regex as re
from collections import defaultdict
import psutil
from typing import BinaryIO

In [129]:
class Tokenizer(ABC):
    """分词器的抽象类，规范了分词器必须要有的方法: `encode()`、`decode()`"""
    def encode(self, string: str) -> list[int]:
        raise NotImplementedError

    def decode(self, indices: list[int]) -> str:
        raise NotImplementedError

class BPETokenizerParams:
    """定义一个 BPETokenizer 所需的全部内容。
    意思是有了这些参数：`vocab`、`merges`，就能构建一个 BPE 分词器。"""
    vocab: dict[int, bytes]     # index -> bytes
    merges: dict[tuple[int, int], int]  # index1,index2 -> new_index
    def __init__(self, vocab: dict[int, bytes], merges: dict[tuple[int, int], int]):
        self.vocab = vocab
        self.merges = merges

def merge(indices: list[int], pair: tuple[int, int], new_index: int) -> list[int]:
    """遍历所给的 `indices` 列表, 主要作用就是更新索引序列（或者说 Token 序列），
    把其中出现的指定 token 对 pair 生成一个新的 token"""
    new_indices = []
    i = 0
    while i < len(indices):
        # i + 1 < len(indices) 是用来保证 i 指向的是列表中的第二个 index
        # indices[i] == pair[0] and indices[i + 1] == pair[1] ：指定的 token 对 pair
        if i + 1 < len(indices) and indices[i] == pair[0] and indices[i + 1] == pair[1]:
            new_indices.append(new_index)
            i += 2
        else:
            # 没有被指定 pair 对的时候，将原来indices中的indice直接添加到new_indices
            new_indices.append(indices[i])  
            i += 1
    return new_indices


In [22]:
# merge() 举一个例子
indices = [1, 2, 3, 2, 3, 4, 2, 3]
pair = (2, 3)
new_index = 99

result = merge(indices, pair, new_index)
print(result)  # [1, 99, 99, 4]

[1, 99, 99, 4, 99]


In [112]:
class BPETokenizer(Tokenizer):
    """定义一个 BPETokenizer 类，继承了 Tokenizer 类，定义了两个方法：`encode()`, `decode()`"""
    def __init__(self, params: BPETokenizerParams):
        self.params = params
        
    def encode(self, string: str) -> list[int]:
        indices = list(map(int, string.encode("utf-8")))
        # Note: this is a very slow implementation
        for pair, new_index in self.params.merges.items():
            indices = merge(indices, pair, new_index)
        return indices
    
    def decode(self, indices: list[int]) -> str:
        bytes_list = list(map(self.params.vocab.get, indices))
        string = b"".join(bytes_list).decode("utf-8")
        return string

In [111]:
def train_bpe(string: str, num_merges: int) -> BPETokenizerParams:
    '''训练 bpe 分词器，
    string: 输入一段字符串
    num_merges: 指定进行几次合并
    返回类型是 BPETokenizerParams '''
    
    # 把字符串编码成 UTF-8 字节序列，再转成整型列表。
    # map() 的作用是：将 string.encode("utf-8")  的结果转换成整数类型
    indices = list(map(int, string.encode("utf-8")))
    
    # merges 定义数据类型是 dict[tuple[int, int], int]，记录的是每次 merge 对应的两个
    merges: dict[tuple[int, int], int] = {}
    
    # 词表 vocab 定义数据类型是 dict[int, bytes]，这里首先初始化了词表
    vocab: dict[int, bytes] = {x: bytes([x]) for x in range(256)}
    
    for i in range(num_merges):
        
        # 统计每一对 token 出现的次数
        # defaultdict(int) 的作用是加入字典的键不存在时，自动创建并计数为0
        counts = defaultdict(int)
        
        # 遍历生成相邻两个 token 的组合（index1, index2）
        for index1, index2 in zip(indices, indices[1:]):
            counts[(index1, index2)] += 1
        
        # 找到 counts 字典中，值最大的键。这里：如果有多个最大值，返回字典顺序下的第一个。
        pair = max(counts, key=counts.get)
        index1, index2 = pair
        
        # merges 更新
        new_index = 256 + i  # i 是从 0 开始的
        merges[pair] = new_index
        
        # 词表更新。两个字节类型的元素相加：不是数值相加，是两个字节拼接到一起
        vocab[new_index] = vocab[index1] + vocab[index2]
        
        # 更新索引序列
        indices = merge(indices, pair, new_index)
        
    return BPETokenizerParams(vocab=vocab, merges=merges)

# 草稿3：实现run_train_bpe

In [1]:
import os 
import sys
from abc import ABC
import regex as re
from collections import defaultdict
import psutil
from typing import BinaryIO

def memory():
    '''查看内存占用'''
    mem = psutil.virtual_memory()
    print(f"可用内存: {mem.available / 1024 / 1024:.2f} MB")
    print(f"内存使用率: {mem.percent}%")

In [3]:
import os 
import regex as re
from collections import defaultdict

def run_train_bpe(
    input_path: str | os.PathLike,
    vocab_size: int,
    special_tokens: list[str],
    **kwargs,
) -> tuple[dict[int, bytes], list[tuple[bytes, bytes]]]:
    """
    给定输入语料的路径，训练一个 BPE 分词器，并输出其 vocab 和 merges 。
    参数：
        input_path (str | os.PathLike)：BPE 分词器训练数据的路径。
        vocab_size (int)：分词器词表的总大小（包括特殊 token）。
        special_tokens (list[str])：一个字符串列表，表示要加入词表的特殊 token。
            这些特殊 token 永远不会被拆分成多个 token，总是保持为一个整体。
            如果这些特殊 token 出现在 `input_path` 中，它们会被视作普通字符串处理。
    返回：
        tuple[dict[int, bytes], list[tuple[bytes, bytes]]]：
            vocab：
                训练得到的分词器词表，字典的 key 是 int 结构（词表中的 token ID），
                value 是 bytes（对应的 token 字节串）。
            merges：
                BPE 合并规则。列表中的每一项是一个 bytes 元组 (<token1>, <token2>)，
                表示 <token1> 和 <token2> 被合并为一个新 token。合并规则按创建顺序排列。
    """
    
    # 1. 词表初始化: 256个基础词、特殊 tokens
    # 词表 vocab 定义数据类型是 dict[int, bytes]，这里首先初始化了词表
    vocab: dict[int, bytes] = {x: bytes([x]) for x in range(256)}
    # 256个基础词，0-255，所以下一个是256
    next_token_id = 256
    # 将特殊 tokens 转换成 byte 格式并加入词表
    for special_token in special_tokens:
        # 注意这里所给的特殊 tokens 为 str 格式，vocab 中的是字节形式的
        vocab[next_token_id] = special_token.encode("utf-8")
        next_token_id += 1
    
    # 2. 预分词
    # 读取数据
    with open(input_path, "r", encoding="utf-8") as f:
        content = f.read()
    # 预分词规则：gpt2 的分词规则
    # 就这个正则化的错误让我改了一天的bug
    # PAT = r"""'(?:[sdmt]|ll|ve|re)| ?\p{{L}}+| ?\p{{N}}+| ?[^\s\p{{L}}\p{{N}}]+|\s+(?!\S)|\s+"""
    PAT = r"""'(?:[sdmt]|ll|ve|re)| ?\p{L}+| ?\p{N}+| ?[^\s\p{L}\p{N}]+|\s+(?!\S)|\s+"""
    # 按照特殊 tokens 分割文档
    texts = re.split("|".join(map(re.escape, special_tokens)), content)
    
    # 对每个文档中的每个部分进行统计
    pre_indices = defaultdict(int)
    for text in texts:
        # 关于 re.finditer() 的使用在 `assignment.md` 中的 `其他` 部分有介绍
        pre_token_matches = re.finditer(PAT, text)
        # 统计分词出现的的次数
        for pre_token_matche in pre_token_matches:
            pre_indices_key = tuple([s.encode() for s in list(pre_token_matche.group())])
            pre_indices[pre_indices_key] += 1

    # 3. BPE 合并
    # 合并次数为词表大小减去初始化的词表
    num_merges = vocab_size - 256 - len(special_tokens)
    # merges：BPE 合并规则。
    merges: list[tuple[bytes, bytes]] = []
    indices = pre_indices.copy()  # 这个相当于换个名字，没什么特别的功能
    
    # 开始合并，每一次循环就是一次合并
    for i in range(num_merges):
        # counts 用来给(<token1>, <token2>)计数
        counts = defaultdict(int)
        # index 是 indices 字典中的 key
        for index in indices:
            # 生成相邻两个 token 的组合（index1, index2）
            for index1, index2 in zip(index, index[1:]):
                # indices[indice] 为合并前（index1, index2）出现的次数
                counts[(index1, index2)] += indices[index]
        
        # 找到出现次数最多的
        # pair = max(counts, key=counts.get)
        # 1. 先找出最大的值
        max_val = max(counts.values())
        # 2. 找出值等于最大值的所有键，然后max()找到字典序最大的合并
        pair = max([k for k, v in counts.items() if v == max_val])
        
        # merges 更新
        index1, index2 = pair
        merges.append(pair)
        # 词表更新。将两个字节相加
        # 两个字节类型的元素相加：不是数值相加，是两个字节拼接到一起
        vocab[next_token_id] = index1 + index2
        next_token_id += 1
        
        # 更新 indices 字典
        new_indices = defaultdict(int)
        for index in indices:
            new_index = []
            index_value = indices[index]
            i = 0
            while i < len(index):
                # i + 1 < len(index) 是用来保证 i 指向的是列表中的第二个 index
                # index[i] == pair[0] and index[i + 1] == pair[1] ：指定的 token 对 pair
                if i + 1 < len(index) and index[i] == pair[0] and index[i + 1] == pair[1]:
                    new_index.append(pair[0] + pair[1])
                    i += 2
                else:
                    # 没有被指定 pair 对的时候，将原来index中的indice直接添加到new_index
                    new_index.append(index[i])
                    i += 1
            new_indices[tuple(new_index)] = index_value
        
        indices = new_indices

    return vocab, merges

In [19]:
vocab, merges = run_train_bpe('../data/TinyStoriesV2-GPT4-valid.txt', 500, ['<|endoftext|>'])

In [ ]:
import platform
import psutil
import torch
import cpuinfo

def system_info():
    cpu_model = cpuinfo.get_cpu_info().get("brand_raw", platform.processor())
    freq = psutil.cpu_freq()

    info = {
        "操作系统": platform.platform(),
        "系统架构": platform.machine(),
        "CPU 型号": cpu_model,
        "CPU 主频": f"{freq.current:.2f} MHz" if freq else "未知",
        "CPU 物理核心数": psutil.cpu_count(logical=False),
        "CPU 逻辑核心数": psutil.cpu_count(logical=True),
    }

    # GPU 信息
    gpu_count = torch.cuda.device_count()
    for i in range(gpu_count):
        props = torch.cuda.get_device_properties(i)
        info[f"GPU {i} 型号"] = torch.cuda.get_device_name(i)
        info[f"GPU {i} 显存总量"] = f"{props.total_memory // (1024**2)} MB"
        info[f"GPU {i} CUDA 核心数"] = props.multi_processor_count
        info[f"GPU {i} 计算能力"] = f"{props.major}.{props.minor}"
        
    return info


if __name__ == "__main__":
    info = system_info()
    print("===============电脑配置信息===============")
    for k, v in info.items():
        print(f"{k}: {v}")


===============电脑配置信息===============
操作系统: Windows-10-10.0.19045-SP0
系统架构: AMD64
CPU 型号: AMD Ryzen 5 5600H with Radeon Graphics
CPU 主频: 3301.00 MHz
CPU 物理核心数: 6
CPU 逻辑核心数: 12
GPU 0 型号: NVIDIA GeForce GTX 1650
GPU 0 显存总量: 4095 MB
GPU 0 CUDA 核心数: 14
GPU 0 计算能力: 7.5


In [10]:
# 草稿
from collections import defaultdict
import regex as re


special_tokens = ['<|endoftext|>']
merges: list[tuple[bytes, bytes]] = []
vocab: dict[int, bytes] = {x: bytes([x]) for x in range(256)}

next_token_id = 256
for special_token in special_tokens:  # 特殊 tokens 
    # 注意这里所给的特殊 tokens 为 str 格式，vocab 中的是字节形式的
    vocab[next_token_id] = special_token.encode("utf-8")
    next_token_id += 1
    
with open('../data/text_example.txt', "r", encoding="utf-8") as f:
    content = f.read()[:10000]
    
# 按照特殊 tokens 分割文档
texts = re.split("|".join(map(re.escape, special_tokens)), content)

pre_indices = defaultdict(int)

special_pat = "|".join(re.escape(token) for token in special_tokens)
# PAT = r"""'(?:[sdmt]|ll|ve|re)| ?\p{{L}}+| ?\p{{N}}+| ?[^\s\p{{L}}\p{{N}}]+|\s+(?!\S)|\s+"""
PAT = r"""'(?:[sdmt]|ll|ve|re)| ?\p{L}+| ?\p{N}+| ?[^\s\p{L}\p{N}]+|\s+(?!\S)|\s+"""

# 对每个文档中的每个部分进行统计
pre_indices = defaultdict(int)

for text in texts:
    # 关于 re.finditer() 的使用在 `assignment.md` 中的 `其他` 部分有介绍
    pre_token_matches = re.finditer(PAT, text)
    # 统计分词出现的的次数
    for pre_token_matche in pre_token_matches:
        # list()能将字符串转换成一个一个字符的列表
        # print(tuple(pre_token_matche.group()))
        pre_indices_key = tuple([bytes([x]) for x in tuple(pre_token_matche.group().encode())])
        pre_indices[pre_indices_key] += 1

indices = pre_indices.copy()

In [11]:
counts = defaultdict(int)

# indice 字典中的 key
for indice in indices:
    # 生成相邻两个 token 的组合（index1, index2）
    for index1, index2 in zip(indice, indice[1:]):
        # indices[indice] 为合并前（index1, index2）出现的次数
        counts[(index1, index2)] += indices[indice]

In [12]:
# 找到出现次数最多的
# pair = max(counts, key=counts.get)  # 这种方式不行！用下面的方法
# 1. 先找出最大的值
max_val = max(counts.values())
# 2. 找出值等于最大值的所有键，然后max()找到字典序最大的合并
pair = max([k for k, v in counts.items() if v == max_val])

# 词表更新。两个字节类型的元素相加：不是数值相加，是两个字节拼接到一起
index1, index2 = pair
vocab[next_token_id] = index1 + index2
next_token_id += 1

# 添加到 merges 中
merges.append(pair)
vocab

{0: b'\x00',
 1: b'\x01',
 2: b'\x02',
 3: b'\x03',
 4: b'\x04',
 5: b'\x05',
 6: b'\x06',
 7: b'\x07',
 8: b'\x08',
 9: b'\t',
 10: b'\n',
 11: b'\x0b',
 12: b'\x0c',
 13: b'\r',
 14: b'\x0e',
 15: b'\x0f',
 16: b'\x10',
 17: b'\x11',
 18: b'\x12',
 19: b'\x13',
 20: b'\x14',
 21: b'\x15',
 22: b'\x16',
 23: b'\x17',
 24: b'\x18',
 25: b'\x19',
 26: b'\x1a',
 27: b'\x1b',
 28: b'\x1c',
 29: b'\x1d',
 30: b'\x1e',
 31: b'\x1f',
 32: b' ',
 33: b'!',
 34: b'"',
 35: b'#',
 36: b'$',
 37: b'%',
 38: b'&',
 39: b"'",
 40: b'(',
 41: b')',
 42: b'*',
 43: b'+',
 44: b',',
 45: b'-',
 46: b'.',
 47: b'/',
 48: b'0',
 49: b'1',
 50: b'2',
 51: b'3',
 52: b'4',
 53: b'5',
 54: b'6',
 55: b'7',
 56: b'8',
 57: b'9',
 58: b':',
 59: b';',
 60: b'<',
 61: b'=',
 62: b'>',
 63: b'?',
 64: b'@',
 65: b'A',
 66: b'B',
 67: b'C',
 68: b'D',
 69: b'E',
 70: b'F',
 71: b'G',
 72: b'H',
 73: b'I',
 74: b'J',
 75: b'K',
 76: b'L',
 77: b'M',
 78: b'N',
 79: b'O',
 80: b'P',
 81: b'Q',
 82: b'R',
 83: b'

In [358]:
# 更新 indices 字典
new_indices = defaultdict(int)

for index in indices:
    index_value = indices[index]
    new_index = []
    i = 0
    while i < len(index):
        # index[i] == pair[0] and index[i + 1] == pair[1] ：指定的 token 对 pair
        if i + 1 < len(index) and index[i] == pair[0] and index[i + 1] == pair[1]:
            new_index.append(pair[0] + pair[1])
            i += 2
            print(new_index)
        else:
            # 没有被指定 pair 对的时候，将原来index中的indice直接添加到new_index
            new_index.append(index[i])  
            i += 1
    new_indices[tuple(new_index)] = index_value

indices = new_indices.copy()

In [262]:
from typing import Dict, Tuple

In [263]:
special_tokens = ['<|endoftext|>']
input_path = '../data/text_example.txt'
vocab_size = 300

# Step 1: Initialize Vocabulary
vocab: Dict[int, bytes] = {i: bytes([i]) for i in range(256)}
next_id = 256
special_token_bytes = [token.encode("utf-8") for token in special_tokens]
for token_bytes in special_token_bytes:
    if token_bytes not in vocab.values():
        vocab[next_id] = token_bytes
        next_id += 1
        
special_token_bytes

[b'<|endoftext|>']

In [264]:
# Step 2: Pre-tokenization
pre_tokens_cnt = defaultdict(int)
def to_bytes_tuple(word: str) -> Tuple[bytes]:
    l = list(tuple(word.encode("utf-8")))
    l = [bytes([x]) for x in l]
    return tuple(l)

with open(input_path, "r", encoding="utf-8") as f:
    text = f.read()

chunks = re.split("|".join(map(re.escape, special_tokens)), text)

for chunk in chunks:
    for m in re.finditer(PAT, chunk):
        word = m.group(0)
        pre_tokens_cnt[to_bytes_tuple(word)] += 1
        
pre_tokens_cnt

defaultdict(int,
            {(b'l', b'o', b'w'): 2,
             (b' ', b'l', b'o', b'w'): 8,
             (b'\n',): 8,
             (b'l', b'o', b'w', b'e', b'r'): 2,
             (b' ', b'l', b'o', b'w', b'e', b'r'): 2,
             (b' ', b'w', b'i', b'd', b'e', b's', b't', b'.'): 6,
             (b'n', b'e', b'w', b'e', b's', b't', b'.'): 2,
             (b' ', b'n', b'e', b'w', b'e', b's', b't', b'.'): 10})

In [265]:
# Step 3: Compute BPE Merges
merges = []

In [266]:
pair_counts = defaultdict(int)
# Count all adjacent byte pairs
for token, cnt in pre_tokens_cnt.items():
    for i in range(len(token) - 1):
        pair = (token[i], token[i + 1])
        pair_counts[pair] += cnt
pair_counts


defaultdict(int,
            {(b'l', b'o'): 14,
             (b'o', b'w'): 14,
             (b' ', b'l'): 10,
             (b'w', b'e'): 16,
             (b'e', b'r'): 4,
             (b' ', b'w'): 6,
             (b'w', b'i'): 6,
             (b'i', b'd'): 6,
             (b'd', b'e'): 6,
             (b'e', b's'): 18,
             (b's', b't'): 18,
             (b't', b'.'): 18,
             (b'n', b'e'): 12,
             (b'e', b'w'): 12,
             (b' ', b'n'): 10})

In [267]:
# Find the most frequent pair(s)
max_count = max(pair_counts.values())
candidates = [k for k, v in pair_counts.items() if v == max_count]
best_pair = max(candidates)
a, b = best_pair

In [268]:
best_pair

(b't', b'.')

In [269]:
# Create new token
new_token = a + b
vocab[next_id] = new_token
next_id += 1

new_token

b't.'

In [270]:
# Apply the merge to all pre-tokenized sequences
# 收集变更
changes = []
for token, cnt in pre_tokens_cnt.items():
    # Find all occurrences of the `best_pair` in `token`
    indices = [i for i in range(len(token) - 1) if token[i:i + 2] == best_pair]
    if indices:
        # Replace each occurrence with `new_token`
        new_pre_token = []
        i = 0
        while i < len(token):
            if i in indices:
                new_pre_token.append(new_token)
                i += 2
            else:
                new_pre_token.append(token[i])
                i += 1
        new_pre_token = tuple(new_pre_token)
        changes.append((token, new_pre_token, cnt))
        
pre_tokens_cnt

defaultdict(int,
            {(b'l', b'o', b'w'): 2,
             (b' ', b'l', b'o', b'w'): 8,
             (b'\n',): 8,
             (b'l', b'o', b'w', b'e', b'r'): 2,
             (b' ', b'l', b'o', b'w', b'e', b'r'): 2,
             (b' ', b'w', b'i', b'd', b'e', b's', b't', b'.'): 6,
             (b'n', b'e', b'w', b'e', b's', b't', b'.'): 2,
             (b' ', b'n', b'e', b'w', b'e', b's', b't', b'.'): 10})

In [271]:
changes

[((b' ', b'w', b'i', b'd', b'e', b's', b't', b'.'),
  (b' ', b'w', b'i', b'd', b'e', b's', b't.'),
  6),
 ((b'n', b'e', b'w', b'e', b's', b't', b'.'),
  (b'n', b'e', b'w', b'e', b's', b't.'),
  2),
 ((b' ', b'n', b'e', b'w', b'e', b's', b't', b'.'),
  (b' ', b'n', b'e', b'w', b'e', b's', b't.'),
  10)]

In [272]:
# 应用变更
for old_token, new_pre_token, cnt in changes:
    pre_tokens_cnt[new_pre_token] = pre_tokens_cnt.get(new_pre_token, 0) + cnt
    del pre_tokens_cnt[old_token]
# Record the merge
merges.append((a, b))

pre_tokens_cnt

defaultdict(int,
            {(b'l', b'o', b'w'): 2,
             (b' ', b'l', b'o', b'w'): 8,
             (b'\n',): 8,
             (b'l', b'o', b'w', b'e', b'r'): 2,
             (b' ', b'l', b'o', b'w', b'e', b'r'): 2,
             (b' ', b'w', b'i', b'd', b'e', b's', b't.'): 6,
             (b'n', b'e', b'w', b'e', b's', b't.'): 2,
             (b' ', b'n', b'e', b'w', b'e', b's', b't.'): 10})

In [273]:
merges

[(b't', b'.')]

# 草稿：尝试优化一下 `run_train_bpe()`

In [ ]:
import os
import regex as re
from collections import defaultdict,Counter
from multiprocessing import Pool
from typing import BinaryIO

def find_chunk_boundaries(
    file: BinaryIO,
    desired_num_chunks: int,
    split_special_token: bytes,
) -> list[int]:
    """
    Chunk the file into parts that can be counted independently.
    May return fewer chunks if the boundaries end up overlapping.
    """
    assert isinstance(split_special_token, bytes), "Must represent special token as a bytestring"

    # Get total file size in bytes
    file.seek(0, os.SEEK_END)
    file_size = file.tell()
    file.seek(0)

    chunk_size = file_size // desired_num_chunks

    # Initial guesses for chunk boundary locations, uniformly spaced
    # Chunks start on previous index, don't include last index
    chunk_boundaries = [i * chunk_size for i in range(desired_num_chunks + 1)]
    chunk_boundaries[-1] = file_size

    mini_chunk_size = 4096  # Read ahead by 4k bytes at a time

    for bi in range(1, len(chunk_boundaries) - 1):
        initial_position = chunk_boundaries[bi]
        file.seek(initial_position)  # Start at boundary guess
        while True:
            mini_chunk = file.read(mini_chunk_size)  # Read a mini chunk

            # If EOF, this boundary should be at the end of the file
            if mini_chunk == b"":
                chunk_boundaries[bi] = file_size
                break

            # Find the special token in the mini chunk
            found_at = mini_chunk.find(split_special_token)
            if found_at != -1:
                chunk_boundaries[bi] = initial_position + found_at
                break
            initial_position += mini_chunk_size

    # Make sure all boundaries are unique, but might be fewer than desired_num_chunks
    return sorted(set(chunk_boundaries))

def pre_count_indices(
    content: str, 
    special_tokens: list
    ) -> defaultdict[tuple[int, ...], int]:
    '''
    对输入的文档进行预分词
    '''
    # 按照特殊 tokens 分割文档
    texts = re.split("|".join(map(re.escape, special_tokens)), content)
    PAT = r"""'(?:[sdmt]|ll|ve|re)| ?\p{L}+| ?\p{N}+| ?[^\s\p{L}\p{N}]+|\s+(?!\S)|\s+"""
    # 对每个文档中的每个部分进行统计
    pre_indices = defaultdict(int)
    for text in texts:
        # 关于 re.finditer() 的使用在 `assignment.md` 中的 `其他` 部分有介绍
        pre_token_matches = re.finditer(PAT, text)
        # 统计分词出现的的次数
        for pre_token_matche in pre_token_matches:
            pre_indices_key = tuple([bytes([x]) for x in tuple(pre_token_matche.group().encode())])
            pre_indices[pre_indices_key] += 1
    return pre_indices

def multi_process_pre_token(input_path, num_processes, special_tokens):
    '''
    并行化预分词阶段
    '''
    # 首先将文本分段
    chunks = []
    with open(input_path, "rb") as f:
        boundaries = find_chunk_boundaries(f, num_processes, b"<|endoftext|>")
        for start, end in zip(boundaries[:-1], boundaries[1:]):
            f.seek(start)
            chunk = f.read(end - start).decode("utf-8", errors="ignore").replace("\r\n", "\n")
            chunks.append(chunk)
            
    # 使用进程池并行处理
    with Pool() as pool:
        dicts = pool.starmap(pre_count_indices, [(chunk, special_tokens) for chunk in chunks])
    
    # 合并结果
    indices = Counter()
    for d in dicts:
        indices.update(d)
    # 转回普通字典
    indices = dict(indices)

    return indices

def update_indices(
    indices: dict[tuple[int, ...], int], 
    pair: tuple[int, int], 
) -> defaultdict[tuple[int, ...], int]:
    '''
    更新 indices 序列的函数
    '''
    new_indices = defaultdict(int)
    for index in indices:
        new_index = []
        index_value = indices[index]
        i = 0
        while i < len(index):
            # i + 1 < len(index) 是用来保证 i 指向的是列表中的第二个 index
            # index[i] == pair[0] and index[i + 1] == pair[1] ：指定的 token 对 pair
            if i + 1 < len(index) and index[i] == pair[0] and index[i + 1] == pair[1]:
                new_index.append(pair[0] + pair[1])
                i += 2
            else:
                # 没有被指定 pair 对的时候，将原来index中的indice直接添加到new_index
                new_index.append(index[i])
                i += 1
        new_indices[tuple(new_index)] = index_value

    return new_indices

def max_pair(
    indices: defaultdict[tuple[int, ...], int]
    )-> tuple:
    '''
    找出出现次数最多的 pair
    '''
    counts = defaultdict(int)  # 用来计数的字典
    for index in indices:
        for index1, index2 in zip(index, index[1:]):
            counts[(index1, index2)] += indices[index]
    max_val = max(counts.values())  # 出现次数最多
    pair = max([k for k, v in counts.items() if v == max_val])  # 字典序最大
    
    return pair

def run_train_bpe(
    input_path: str | os.PathLike,
    vocab_size: int,
    special_tokens: list[str],
    **kwargs,
) -> tuple[dict[int, bytes], list[tuple[bytes, bytes]]]:
    """
    给定输入语料的路径，训练一个 BPE 分词器，并输出其 vocab 和 merges 。
    参数：
        input_path (str | os.PathLike)：BPE 分词器训练数据的路径。
        vocab_size (int)：分词器词表的总大小（包括特殊 token）。
        special_tokens (list[str])：一个字符串列表，表示要加入词表的特殊 token。
            这些特殊 token 永远不会被拆分成多个 token，总是保持为一个整体。
            如果这些特殊 token 出现在 `input_path` 中，它们会被视作普通字符串处理。
    返回：
        tuple[dict[int, bytes], list[tuple[bytes, bytes]]]：
            vocab：
                训练得到的分词器词表，字典的 key 是 int 结构（词表中的 token ID），
                value 是 bytes（对应的 token 字节串）。
            merges：
                BPE 合并规则。列表中的每一项是一个 bytes 元组 (<token1>, <token2>)，
                表示 <token1> 和 <token2> 被合并为一个新 token。合并规则按创建顺序排列。
    """
    
    # 1. 初始化: 256个基础词、特殊 tokens
    vocab: dict[int, bytes] = {x: bytes([x]) for x in range(256)}
    merges: list[tuple[bytes, bytes]] = []
    next_token_id = 256
    # 将特殊 tokens 转换成 byte 格式并加入词表
    for special_token in special_tokens:
        vocab[next_token_id] = special_token.encode("utf-8")
        next_token_id += 1

    # 2. 预分词
    indices = multi_process_pre_token(input_path, 4, special_tokens)

    # 3. BPE 合并
    num_merges = vocab_size - 256 - len(special_tokens)
    for i in range(num_merges):
        # 找出出现次数最多的 pair
        pair = max_pair(indices)
        # merges 更新
        merges.append(pair)
        # 词表更新。
        vocab[next_token_id] = pair[0] + pair[1]
        next_token_id += 1
        # indices 序列更新
        indices = update_indices(indices, pair)

    return vocab, merges

In [ ]:
vocab, merges = run_train_bpe('../data/TinyStoriesV2-GPT4-valid.txt', 500, ['<|endoftext|>'])

In [4]:
# 直接打开的文本
with open('../data/TinyStoriesV2-GPT4-valid.txt', "r") as f:
    direct_text = f.read()
    ceshi = pre_count_indices(f.read(), ['<|endoftext|>'])
    
# 使用分段后的文本（整个部分）
chunks = ''
with open('../data/TinyStoriesV2-GPT4-valid.txt', "rb") as f:
    boundaries = find_chunk_boundaries(f, 4, b"<|endoftext|>")
    for start, end in zip(boundaries[:-1], boundaries[1:]):
        f.seek(start)
        chunk = f.read(end - start).decode("utf-8", errors="ignore").replace("\r\n", "\n")
        chunks += chunk
        
print(direct_text[:200])
print('---')
print(chunks[:200])

Spot. Spot saw the shiny car and said, "Wow, Kitty, your car is so bright and clean!" Kitty smiled and replied, "Thank you, Spot. I polish it every day."
After playing with the car, Kitty and Spot fel
---
Spot. Spot saw the shiny car and said, "Wow, Kitty, your car is so bright and clean!" Kitty smiled and replied, "Thank you, Spot. I polish it every day."
After playing with the car, Kitty and Spot fel


## 继续优化

In [ ]:
import os
import regex as re
from collections import defaultdict,Counter
from multiprocessing import Pool
from typing import BinaryIO

def find_chunk_boundaries(
    file: BinaryIO,
    desired_num_chunks: int,
    split_special_token: bytes,
) -> list[int]:
    """
    Chunk the file into parts that can be counted independently.
    May return fewer chunks if the boundaries end up overlapping.
    """
    assert isinstance(split_special_token, bytes), "Must represent special token as a bytestring"

    # Get total file size in bytes
    file.seek(0, os.SEEK_END)
    file_size = file.tell()
    file.seek(0)

    chunk_size = file_size // desired_num_chunks

    # Initial guesses for chunk boundary locations, uniformly spaced
    # Chunks start on previous index, don't include last index
    chunk_boundaries = [i * chunk_size for i in range(desired_num_chunks + 1)]
    chunk_boundaries[-1] = file_size

    mini_chunk_size = 4096  # Read ahead by 4k bytes at a time

    for bi in range(1, len(chunk_boundaries) - 1):
        initial_position = chunk_boundaries[bi]
        file.seek(initial_position)  # Start at boundary guess
        while True:
            mini_chunk = file.read(mini_chunk_size)  # Read a mini chunk

            # If EOF, this boundary should be at the end of the file
            if mini_chunk == b"":
                chunk_boundaries[bi] = file_size
                break

            # Find the special token in the mini chunk
            found_at = mini_chunk.find(split_special_token)
            if found_at != -1:
                chunk_boundaries[bi] = initial_position + found_at
                break
            initial_position += mini_chunk_size

    # Make sure all boundaries are unique, but might be fewer than desired_num_chunks
    return sorted(set(chunk_boundaries))

def pre_count_indices(
    content: str, 
    special_tokens: list
    ) -> defaultdict[tuple[int, ...], int]:
    '''
    对输入的文档进行预分词
    '''
    # 按照特殊 tokens 分割文档
    texts = re.split("|".join(map(re.escape, special_tokens)), content)
    PAT = r"""'(?:[sdmt]|ll|ve|re)| ?\p{L}+| ?\p{N}+| ?[^\s\p{L}\p{N}]+|\s+(?!\S)|\s+"""
    # 对每个文档中的每个部分进行统计
    pre_indices = defaultdict(int)
    for text in texts:
        # 关于 re.finditer() 的使用在 `assignment.md` 中的 `其他` 部分有介绍
        pre_token_matches = re.finditer(PAT, text)
        # 统计分词出现的的次数
        for pre_token_matche in pre_token_matches:
            pre_indices_key = tuple([bytes([x]) for x in tuple(pre_token_matche.group().encode())])
            pre_indices[pre_indices_key] += 1
    return pre_indices

def multi_process_pre_token(input_path, num_processes, special_tokens):
    '''
    并行化预分词阶段
    '''
    # 首先将文本分段
    chunks = []
    with open(input_path, "rb") as f:
        boundaries = find_chunk_boundaries(f, num_processes, b"<|endoftext|>")
        for start, end in zip(boundaries[:-1], boundaries[1:]):
            f.seek(start)
            chunk = f.read(end - start).decode("utf-8", errors="ignore").replace("\r\n", "\n")
            chunks.append(chunk)
            
    # 使用进程池并行处理
    with Pool() as pool:
        dicts = pool.starmap(pre_count_indices, [(chunk, special_tokens) for chunk in chunks])
    
    # 合并结果
    indices = Counter()
    for d in dicts:
        indices.update(d)
    # 转回普通字典
    indices = dict(indices)

    return indices

def update_indices(
    indices: dict[tuple[int, ...], int], 
    pair: tuple[int, int], 
) -> defaultdict[tuple[int, ...], int]:
    '''
    更新 indices 序列的函数
    '''
    new_indices = defaultdict(int)
    for index in indices:
        new_index = []
        index_value = indices[index]
        i = 0
        while i < len(index):
            # i + 1 < len(index) 是用来保证 i 指向的是列表中的第二个 index
            # index[i] == pair[0] and index[i + 1] == pair[1] ：指定的 token 对 pair
            if i + 1 < len(index) and index[i] == pair[0] and index[i + 1] == pair[1]:
                new_index.append(pair[0] + pair[1])
                i += 2
            else:
                # 没有被指定 pair 对的时候，将原来index中的indice直接添加到new_index
                new_index.append(index[i])
                i += 1
        new_indices[tuple(new_index)] = index_value

    return new_indices

def max_pair(
    indices: defaultdict[tuple[int, ...], int]
    )-> tuple:
    '''
    找出出现次数最多的 pair
    '''
    counts = defaultdict(int)  # 用来计数的字典
    for index in indices:
        for index1, index2 in zip(index, index[1:]):
            counts[(index1, index2)] += indices[index]
    max_val = max(counts.values())  # 出现次数最多
    pair = max([k for k, v in counts.items() if v == max_val])  # 字典序最大
    
    return pair

def run_train_bpe(
    input_path: str | os.PathLike,
    vocab_size: int,
    special_tokens: list[str],
    **kwargs,
) -> tuple[dict[int, bytes], list[tuple[bytes, bytes]]]:
    """
    给定输入语料的路径，训练一个 BPE 分词器，并输出其 vocab 和 merges 。
    参数：
        input_path (str | os.PathLike)：BPE 分词器训练数据的路径。
        vocab_size (int)：分词器词表的总大小（包括特殊 token）。
        special_tokens (list[str])：一个字符串列表，表示要加入词表的特殊 token。
            这些特殊 token 永远不会被拆分成多个 token，总是保持为一个整体。
            如果这些特殊 token 出现在 `input_path` 中，它们会被视作普通字符串处理。
    返回：
        tuple[dict[int, bytes], list[tuple[bytes, bytes]]]：
            vocab：
                训练得到的分词器词表，字典的 key 是 int 结构（词表中的 token ID），
                value 是 bytes（对应的 token 字节串）。
            merges：
                BPE 合并规则。列表中的每一项是一个 bytes 元组 (<token1>, <token2>)，
                表示 <token1> 和 <token2> 被合并为一个新 token。合并规则按创建顺序排列。
    """
    
    # 1. 初始化: 256个基础词、特殊 tokens
    vocab: dict[int, bytes] = {x: bytes([x]) for x in range(256)}
    merges: list[tuple[bytes, bytes]] = []
    next_token_id = 256
    # 将特殊 tokens 转换成 byte 格式并加入词表
    for special_token in special_tokens:
        vocab[next_token_id] = special_token.encode("utf-8")
        next_token_id += 1

    # 2. 预分词
    indices = multi_process_pre_token(input_path, 4, special_tokens)

    # 3. BPE 合并
    num_merges = vocab_size - 256 - len(special_tokens)
    for i in range(num_merges):
        # 找出出现次数最多的 pair
        pair = max_pair(indices)
        # indices 序列更新
        indices = update_indices(indices, pair)
        
        # merges 更新
        merges.append(pair)
        # 词表更新。
        vocab[next_token_id] = pair[0] + pair[1]
        next_token_id += 1

    return vocab, merges